# WDL — Module 6 · Session 30 — Capstone workshop and submission
### Workshop

| | |
|---|---|
| **Course** | Hydroinformatics in a Changing Climate: Water Data Literacy for All |
| **Department** | Civil Engineering · CHRIST (Deemed to be University) |
| **Instructor** | Dr Arpan Pradhan |
| **Module** | Module 6 — From Data to Dashboard |
| **Session** | Session 30 of 30 |
| **CO** | CO5 |
| **Topics** | Finish, submit and peer-review the capstone dashboard |
| **Water hook** | *My District's Water Story · peer review* |

---
> **How to use this notebook.** No coding needed. Read each explanation, then press ▶ on the box below it
> (or click the box and press `Shift + Enter`). Some boxes have sliders, number boxes or dropdowns: change them, then press ▶ again.
> Go to **File → Save a copy in Drive** first, so the notebook is yours.

## Capstone: My District's Water Story
This notebook puts your five capstone steps together into **one shareable page**. Run each step, fill in your words, and
download the finished dashboard. Then submit it on CuX and review two classmates' work.

**You will need:** your district name, its area (km²), IMD's normal rainfall (Module 1), and your notes from the
workshops in Modules 2–5.

In [ ]:
#@title ▶ Step 0 · Get ready (press ▶ once before anything else)
import os, json, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
_orig_bar = plt.Axes.bar
def _bar_int_years(self, x, *a, **k):        # whole-number year ticks on bar charts of years
    out = _orig_bar(self, x, *a, **k)
    try:
        xs = np.asarray(x)
        if xs.dtype.kind in "iu" and xs.min() > 1800: self.xaxis.set_major_locator(MaxNLocator(integer=True))
    except Exception: pass
    return out
plt.Axes.bar = _bar_int_years
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110, "font.size": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, GREY, TEAL, GOLD = "#2C6E9B", "#DD8452", "#C44E52", "#8C8C8C", "#1F7A8C", "#C9962E"
AUTO = os.environ.get("WDL_SELFTEST") == "1"

def ok(msg="Correct!"):  display(Markdown(f"✅ **{msg}**"))
def no(msg="Not quite. Check your working and try again."): display(Markdown(f"❌ {msg}"))
def _num(x):
    try: return float(str(x).replace(",", "").replace("%", "").strip())
    except (TypeError, ValueError): return None
def close(a, b, tol=0.02):
    return a is not None and abs(a - b) <= max(abs(b) * tol, 0.01)
def check_num(ans, exp, tol=0.02, fmt="{:,.1f}", hint=""):
    if AUTO: ans = exp
    v = _num(ans)
    if v is None: no("Type your answer in the box first (numbers only)."); return False
    if close(v, float(exp), tol): ok("Correct: " + fmt.format(exp)); return True
    no("Not quite. " + hint); return False
def check_choice(ans, exp, explain="", hint=""):
    if AUTO: ans = exp
    if ans in ("", "choose"): no("Choose an answer from the dropdown first."); return False
    if ans == exp: ok(("Correct. " + explain).strip()); return True
    no(("Not quite. " + hint).strip()); return False

def make_sample_district():
    """Illustrative, synthetic daily rainfall for a South Indian district, 1981-2024."""
    rng = np.random.default_rng(1981)
    days = pd.date_range("1981-01-01", "2024-12-31", freq="D")
    p_wet = np.array([0.03, 0.03, 0.06, 0.15, 0.30, 0.35, 0.42, 0.42, 0.45, 0.45, 0.28, 0.08])
    mu = np.array([4, 4, 7, 11, 13, 8, 8, 9, 13, 14, 12, 6], float)
    years = np.arange(1981, 2025)
    yf = dict(zip(years, np.exp(rng.normal(0, 0.17, len(years)))))
    m = days.month.values - 1
    t = days.year.values - 1981
    yfac = np.array([yf[y] for y in days.year.values])
    p = np.clip(p_wet[m] * (1 - 0.0025 * t) * yfac ** 0.5, 0, 0.95)
    inten = mu[m] * (1 + 0.005 * t) * yfac ** 0.5
    wet = rng.random(len(days)) < p
    amt = rng.gamma(0.7, inten / 0.7)
    rain = np.round(np.where(wet, amt, 0.0), 1)
    for d, v in [("1991-11-14", 142.0), ("2005-10-24", 186.2), ("2017-08-14", 151.4), ("2022-09-05", 168.9)]:
        rain[days.get_loc(pd.Timestamp(d))] = v
    return pd.Series(rain, index=days, name="rain_mm")

def make_messy_record(sample):
    """Three years of the sample record with typical gauge-record problems added."""
    df = sample["2018":"2020"].rename("rain_mm").reset_index()
    df.columns = ["date", "rain_mm"]
    flag_dates = ["2018-02-03", "2018-05-19", "2018-08-07", "2018-11-25", "2019-01-14",
                  "2019-04-02", "2019-09-18", "2019-12-30", "2020-06-06", "2020-10-11"]
    blank_dates = ["2018-07-21", "2019-03-09", "2020-02-17", "2020-08-28"]
    df.loc[df.date.isin(pd.to_datetime(flag_dates)), "rain_mm"] = -999.0
    df.loc[df.date.isin(pd.to_datetime(blank_dates)), "rain_mm"] = np.nan
    df.loc[df.date == "2018-09-12", "rain_mm"] = 999.9      # typing error
    df.loc[df.date == "2020-04-23", "rain_mm"] = -3.0       # impossible negative
    df = df[~df.date.between("2019-07-01", "2019-07-12")]  # 12 days never recorded
    dup = df[df.date.isin(pd.to_datetime(["2018-06-15", "2019-10-03", "2020-11-20"]))]
    df = pd.concat([df, dup]).sort_values("date", kind="stable").reset_index(drop=True)
    df["date"] = df.date.dt.strftime("%Y-%m-%d")
    return df

SAMPLE_URL = "https://raw.githubusercontent.com/PradhanArpan/waterliteracy/main/data/sample_district_daily_1981_2024.csv"
try:
    sample = pd.read_csv(SAMPLE_URL, parse_dates=["date"], index_col="date")["rain_mm"]
except Exception:
    sample = make_sample_district()   # works offline too: same numbers

def find_place(name):
    """Place name -> (lat, lon, label) using Open-Meteo geocoding. Prefers places in India."""
    parts = [p.strip() for p in str(name).split(",") if p.strip()]
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": parts[0], "count": 10, "language": "en", "format": "json"}, timeout=20)
    res = r.json().get("results", []) or []
    res = [x for x in res if x.get("country_code") == "IN"] or res
    if len(parts) > 1:
        res = [x for x in res if parts[1].lower() in str(x.get("admin1", "")).lower()] or res
    if not res: raise ValueError(f"No place found for '{name}'")
    x = res[0]
    return x["latitude"], x["longitude"], f'{x["name"]}, {x.get("admin1", "")}'.strip(", ")

def fetch_power(lat, lon, start=1981, end=2024):
    """Daily rainfall (mm/day) from NASA POWER (PRECTOTCORR) for one point."""
    r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", timeout=90, params={
        "parameters": "PRECTOTCORR", "community": "AG", "latitude": round(lat, 4), "longitude": round(lon, 4),
        "start": f"{start}0101", "end": f"{end}1231", "format": "JSON"})
    r.raise_for_status()
    d = r.json()["properties"]["parameter"]["PRECTOTCORR"]
    s = pd.Series(d, dtype=float)
    s.index = pd.to_datetime(s.index, format="%Y%m%d")
    s[s < -98] = np.nan
    return s.round(1).rename("rain_mm")

def fetch_era5(lat, lon, start="2021-01-01", end="2021-12-31"):
    """Daily rainfall (mm/day) from the ERA5 reanalysis via Open-Meteo."""
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params={
        "latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
        "daily": "precipitation_sum", "timezone": "Asia/Kolkata"})
    r.raise_for_status()
    d = r.json()["daily"]
    return pd.Series(d["precipitation_sum"], index=pd.to_datetime(d["time"]), dtype=float).rename("rain_mm")

def load_rain(source, place="", lat=0.0, lon=0.0):
    """Returns (daily rainfall Series, label, lat, lon). Falls back to the sample district."""
    if source.startswith("Sample") or AUTO:
        return sample, "Sample district (synthetic)", None, None
    try:
        if not (lat and lon):
            lat, lon, place = find_place(place)
        cache = f"rain_{lat:.3f}_{lon:.3f}.csv"
        if os.path.exists(cache):
            s = pd.read_csv(cache, parse_dates=["date"], index_col="date")["rain_mm"]
        else:
            s = fetch_power(lat, lon)
            s.rename_axis("date").to_csv(cache)
        return s, f"{place} ({lat:.2f}°N, {lon:.2f}°E) · NASA POWER", lat, lon
    except Exception as e:
        display(Markdown(f"⚠️ Could not download data for **{place or 'your place'}** ({type(e).__name__}). "
                         "Using the sample district instead. Check the place name or type latitude and longitude."))
        return sample, "Sample district (synthetic)", None, None

def annual(s):  return s.groupby(s.index.year).sum(min_count=300)
def monsoon(s):
    m = s[s.index.month.isin([6, 7, 8, 9])]
    return m.groupby(m.index.year).sum(min_count=100)
display(Markdown("✅ **Ready.** Tools loaded. Now run **Step 1** to choose your data."))

---
## Step 1 · Choose your data
> **About the data.** *Sample district* is an illustrative, synthetic 44-year record (1981–2024) made for practice;
> it is not an official record, and answers in the practice tasks and the Sheet use it.
> *My place* downloads real daily rainfall for any location from **NASA POWER** (satellite and model based, from 1981).
> Type a town or district name, or enter latitude and longitude.

In [ ]:
#@title ▶ Step 1 · Choose your data { run: "auto" }
data_source = "My place (live download from NASA POWER)" #@param ["Sample district (works offline)", "My place (live download from NASA POWER)"]
place = "Mysuru, Karnataka" #@param {type:"string"}
latitude = 0 #@param {type:"number"}
longitude = 0 #@param {type:"number"}
rain, label, LAT, LON = load_rain(data_source, place, latitude, longitude)
display(Markdown(f"**Data:** {label} · {rain.index.min():%d %b %Y} to {rain.index.max():%d %b %Y} · "
                 f"{len(rain):,} days"))

In [ ]:
#@title ▶ Step 2 · Compute the facts (press ▶)
from scipy import stats
A = annual(rain); SW = monsoon(rain)
NORMAL, NORMAL_SW = A.loc[1991:2020].mean(), SW.loc[1991:2020].mean()
DEP = (SW - NORMAL_SW) / NORMAL_SW * 100
RD = (rain >= 2.5).groupby(rain.index.year).sum().loc[A.index]
SDII = A / RD
MON = rain.groupby([rain.index.year, rain.index.month]).sum().groupby(level=1).mean()
TOP = rain.sort_values(ascending=False).head(5)
def trend(series):
    s = series.dropna(); return stats.theilslopes(s.values, s.index)[0] * 10, stats.kendalltau(s.index, s.values)[1]
def verdict(p, slope):
    return ("Clear increase" if slope > 0 else "Clear decrease") if p < 0.05 else "No clear trend"
def cat(d): return "Large deficient" if d <= -60 else "Deficient" if d <= -20 else "Normal" if d < 20 else "Excess" if d < 60 else "Large excess"
CATC = {"Large deficient": "#8E2C2C", "Deficient": RED, "Normal": GREY, "Excess": BLUE, "Large excess": "#1B3F66"}
PLACE = label.split(" · ")[0]
display(Markdown(f"Facts ready for **{PLACE}**: normal {NORMAL:,.0f} mm/year; latest monsoon ({DEP.index[-1]}) {DEP.iloc[-1]:+.0f}% ({cat(DEP.iloc[-1])})."))

---
## Step 3 · Your words

In [ ]:
#@title ▶ My text { run: "auto" }
district_name = "" #@param {type:"string"}
headline = "" #@param {type:"string"}
data_story_150_words = "" #@param {type:"string"}
what_people_experienced = "" #@param {type:"string"}
data_notes_and_limits = "" #@param {type:"string"}
NAME = district_name or PLACE
if AUTO:
    headline = "The sample district's monsoon swings widely, and rainy days are getting fewer"
    data_story_150_words = "Practice text."
words = len(data_story_150_words.split())
display(Markdown(f"Story length: **{words} words** " + ("✅" if 120 <= words <= 180 else "(aim for 120–180)")))

---
## Step 4 · Build the page

In [ ]:
#@title ▶ Build My District's Water Story
import textwrap
sl_a, p_a = trend(A); sl_r, p_r = trend(RD)
ev = rain.idxmax()
fig = plt.figure(figsize=(16.5, 11.7))      # A3 landscape
gs = fig.add_gridspec(4, 3, height_ratios=[0.32, 1, 1, 0.75], hspace=0.55, wspace=0.28)
t = fig.add_subplot(gs[0, :]); t.axis("off")
for i, (k, v) in enumerate([("Normal year (1991–2020)", f"{NORMAL:,.0f} mm"), (f"Monsoon {DEP.index[-1]} vs normal", f"{DEP.iloc[-1]:+.0f}%"),
                            ("Wettest day on record", f"{rain.max():.0f} mm"), ("Rainy days per year", f"{RD.mean():.0f}"),
                            ("Annual total trend", verdict(p_a, sl_a))]):
    t.text(0.01 + i * 0.2, 0.62, v, fontsize=21, fontweight="bold", color=BLUE, transform=t.transAxes)
    t.text(0.01 + i * 0.2, 0.12, k, fontsize=11, color=GREY, transform=t.transAxes)
a = fig.add_subplot(gs[1, 0]); a.bar(range(1, 13), MON.values, color=BLUE); a.set_xticks(range(1, 13)); a.set_xticklabels("JFMAMJJASOND")
a.set_title("1 · What is normal here? (mm by month)", loc="left", fontsize=12)
a = fig.add_subplot(gs[1, 1]); d = DEP.iloc[-15:]; a.bar(d.index, d.values, color=[CATC[cat(v)] for v in d.values]); a.axhline(0, color="black", lw=0.8)
[a.axhline(y, color=GREY, ls="--", lw=0.8) for y in (-20, 20)]; a.set_title("2 · Recent monsoons (% from normal)", loc="left", fontsize=12)
a = fig.add_subplot(gs[1, 2]); w = rain[ev - pd.Timedelta(days=10): ev + pd.Timedelta(days=10)]
a.bar(w.index, w.values, color=[RED if x == ev else BLUE for x in w.index]); a.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
a.tick_params(axis="x", rotation=30); a.set_title(f"3 · Biggest event: {rain[ev]:.0f} mm on {ev:%d %b %Y}", loc="left", fontsize=12)
for k, (S, nm, c) in enumerate([(A, "Annual total (mm)", BLUE), (RD, "Rainy days", TEAL), (SDII, "Rain per rainy day (mm)", ORANGE)]):
    a = fig.add_subplot(gs[2, k]); s_, i_ = stats.theilslopes(S.values, S.index)[:2]; sl, p = trend(S)
    a.plot(S.index, S.values, "o", color=c, alpha=0.55, ms=4); a.plot(S.index, i_ + s_ * S.index, color="black", lw=2)
    a.set_title(f"4 · {nm}: {verdict(p, sl).lower()}", loc="left", fontsize=12)
tx = fig.add_subplot(gs[3, :]); tx.axis("off")
story = textwrap.fill(data_story_150_words or "(Write your 150-word data story in Step 3.)", 190)
people = textwrap.fill("What people experienced: " + (what_people_experienced or "(add from news or local knowledge)"), 190)
tx.text(0, 1, story, va="top", fontsize=11, transform=tx.transAxes)
tx.text(0, 0.38, people, va="top", fontsize=11, style="italic", transform=tx.transAxes)
tx.text(0, 0.02, f"Data: {label}. Normal = 1991–2020 average; categories: IMD; trends: Sen's slope, Mann–Kendall (p < 0.05 = clear). "
        f"{data_notes_and_limits}", fontsize=9, color=GREY, transform=tx.transAxes)
fig.suptitle(headline or f"{NAME}: water story", fontsize=22, fontweight="bold", x=0.01, ha="left", y=0.995)
fig.text(0.99, 0.995, "My District's Water Story · Water Data Literacy for All · CHRIST CuX", ha="right", va="top", fontsize=9, color=GREY)
fig.savefig("My_District_Water_Story.png", dpi=150, bbox_inches="tight"); fig.savefig("My_District_Water_Story.pdf", bbox_inches="tight")
plt.show()

---
## Step 5 · Download and submit

In [ ]:
#@title ▶ Download my capstone
try:
    from google.colab import files; files.download("My_District_Water_Story.png"); files.download("My_District_Water_Story.pdf")
except Exception:
    display(Markdown("Saved **My_District_Water_Story.png** and **.pdf** in the notebook's files."))
problems = [m for m, bad in [("Using the sample district: download your own district in Step 1", LAT is None and not AUTO),
                             ("Write a headline that states a finding", len(headline.split()) < 6),
                             ("Story should be 120–180 words", not (120 <= len(data_story_150_words.split()) <= 180) and not AUTO)] if bad]
(ok if not problems else no)("Ready to submit." if not problems else "Before submitting: " + "; ".join(problems) + ".")

---
## Peer review rubric (use it on two classmates' dashboards)

| Criterion | Weight | Look for |
|---|---|---|
| Correct use and citation of data | 25% | Source named; units right; own district used |
| Accuracy of analysis | 25% | Numbers match the data; categories and return periods used correctly |
| Honest interpretation of trends and limits | 20% | Says "no clear trend" when p ≥ 0.05; mentions data limits |
| Clarity of visual design | 15% | Finding headline; bars from zero; labelled axes |
| Quality of the data story | 15% | Number first, comparison, meaning for people, source |

**Submit on CuX:** the PNG or PDF, plus the Looker Studio link if you made one. Then review two classmates using this rubric.
Congratulations on finishing the course!